# Landlord Sales Dashboard 01165 — Summarecon Emerald Karawang
>Generate dashboard HTML statis (KPI, combo chart, detail per payment type, filter tanggal) untuk pihak landlord.

Sumber data: `daily_est_income_per_store_for_landlord` (1 query, semua view dihitung di browser)

Template: `dashboard_template.html` → Output: `output/01165_Summarecon_Emerald_Karawang_Sales_Dashboard.html`

Created at : 2026-11-12

Created by lahiardhan@jiwagroup.com

# IMPORT LIBRARY

In [1]:
from datetime import datetime
from pathlib import Path
import json

import mysql.connector
from pytz import timezone

import pandas as pd

import importlib, importlib.util

import warnings
warnings.filterwarnings("ignore")

# CONFIG DB

In [2]:
# Import Config Data
def module_from_file(module_name, file_path):
    spec = importlib.util.spec_from_file_location(module_name, file_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

config_dw = module_from_file("config_dw", "../../db_config/config_db_datawarehouse.py")

# CONFIG DASHBOARD

In [3]:
OUTLET_CODE  = '01165'
OUTLET_TITLE = '01165 SUMMARECON EMERALD KARAWANG'

# Lock L60D: hanya data 60 hari terakhir yang diambil & di-embed ke HTML (dashboard juga mengunci di sisi browser)
LOOKBACK_DAYS = 60

# Outlet baru buka 1 Nov 2026. Transaksi 29-31 Okt = data testing, jadi data tidak pernah diambil sebelum tanggal ini.
DATA_START = '2026-11-01'

# (template, output) - semua dashboard di-generate dari data yang sama
DASHBOARDS = [
    (Path('dashboard_template.html'),             Path('output') / '01165_Summarecon_Emerald_Karawang_Sales_Dashboard.html'),
    (Path('dashboard_template_jiwa_light.html'), Path('output') / '01165_Summarecon_Emerald_Karawang_Sales_Dashboard_JiwaLight.html'),
]

# Urutan & label kolom tabel detail (key = nama kolom di query Metabase ke-3)
PAYMENT_COLUMNS = [
    ('cash',          'Cash'),
    ('bri',           'BRI'),
    ('go_food',       'Go Food'),
    ('grab_food',     'Grab Food'),
    ('grab_dine_out', 'Grab Dine Out'),
    ('shopee_food',   'Shopee Food'),
    ('tokopedia_go',  'Tokopedia Go'),
    ('jiwaplus',      'Jiwa+'),
    ('iseller_qris',  'iSeller QRIS'),
    ('qpon',          'QPON'),
    ('other',         'Other'),
]
BUCKETS = [k for k, _ in PAYMENT_COLUMNS]

# MAIN PROGRESS

### 1. Extract Data

Satu query untuk ketiga view Metabase (KPI, combo chart, detail): grain `date × outlet_name × payment bucket`.
Mapping payment type sama persis dengan query detail; payment type yang belum ter-mapping / NULL masuk ke `other`.

In [4]:
def get_data(outlet_code, start_date):
    conn = mysql.connector.connect(host=config_dw.hostname,
                                   port=3306,
                                   user=config_dw.username,
                                   password=config_dw.password,
                                   db=config_dw.db_target)  # tabel ada di data_warehouse_jiwa
    query = """
        SELECT
            DATE_FORMAT(`date`, '%Y-%m-%d') AS `date`,
            outlet_name,
            CASE
                WHEN payment_type = 'Cash'          THEN 'cash'
                WHEN payment_type = 'BRI'           THEN 'bri'
                WHEN payment_type = 'Go Food'       THEN 'go_food'
                WHEN payment_type = 'Grab Food'     THEN 'grab_food'
                WHEN payment_type = 'Grab Dine Out' THEN 'grab_dine_out'
                WHEN payment_type = 'Shopee Food'   THEN 'shopee_food'
                WHEN payment_type = 'Tokopedia Go'  THEN 'tokopedia_go'
                WHEN payment_type = 'Jiwa+'         THEN 'jiwaplus'
                WHEN payment_type IN ('iSeller QRIS', 'iseller_qris', 'isellerpay_qris', 'iSeller Pay QRIS') THEN 'iseller_qris'
                WHEN payment_type = 'QPON'          THEN 'qpon'
                ELSE 'other'
            END AS bucket,
            SUM(total_order) AS total_order,
            SUM(subtotal - COALESCE(sales_type_commission, 0)) AS income
        FROM daily_est_income_per_store_for_landlord
        WHERE outlet_code = '{0}'
            AND `date` >= '{1}'
        GROUP BY 1, 2, 3
        ORDER BY 1
        """.format(outlet_code, start_date)

    df = pd.read_sql(query, conn)
    conn.close()
    return df

today_jkt  = datetime.now(timezone('Asia/Jakarta')).date()
start_date = max((today_jkt - pd.Timedelta(days=LOOKBACK_DAYS)).strftime('%Y-%m-%d'), DATA_START)
print('Lock L60D + data start :', start_date, '->', today_jkt)

raw = get_data(OUTLET_CODE, start_date)
print(raw.shape)
raw.head()

Lock L60D + data start : 2026-11-01 -> 2026-11-16
(133, 5)


,date,outlet_name,bucket,total_order,income
0,2026-11-01,01165 SUMMARECON EMERALD KARAWANG KWG,cash,6.0,2.063636e+05
1,2026-11-01,01165 SUMMARECON EMERALD KARAWANG KWG,go_food,6.0,2.861874e+05
2,2026-11-01,01165 SUMMARECON EMERALD KARAWANG KWG,grab_food,5.0,2.320418e+05
3,2026-11-01,01165 SUMMARECON EMERALD KARAWANG KWG,iseller_qris,88.0,4.503540e+06
4,2026-11-01,01165 SUMMARECON EMERALD KARAWANG KWG,jiwaplus,53.0,1.902302e+06


### 2. Transform

Pivot jadi 1 baris per (`date`, `outlet_name`) — sama dengan query detail Metabase. `daily_income` = jumlah semua kolom payment.

In [5]:
df = raw.copy()
df['outlet_name'] = df['outlet_name'].fillna('')
df['total_order'] = pd.to_numeric(df['total_order'], errors='coerce').fillna(0).astype(float)
df['income']      = pd.to_numeric(df['income'], errors='coerce').fillna(0).astype(float)

pivot = (df.pivot_table(index=['date', 'outlet_name'], columns='bucket', values='income',
                        aggfunc='sum', fill_value=0)
           .reindex(columns=BUCKETS, fill_value=0))
orders = df.groupby(['date', 'outlet_name'])['total_order'].sum()

detail = pivot.join(orders).reset_index().sort_values('date').reset_index(drop=True)
detail['daily_income'] = detail[BUCKETS].sum(axis=1)
detail = detail[['date', 'outlet_name', 'total_order'] + BUCKETS + ['daily_income']]

print('Rows      :', len(detail))
print('Date range:', detail['date'].min(), '->', detail['date'].max())
detail.tail()

Rows      : 15
Date range: 2026-11-01 -> 2026-11-15


,date,outlet_name,total_order,cash,bri,go_food,grab_food,grab_dine_out,shopee_food,tokopedia_go,jiwaplus,iseller_qris,qpon,other,daily_income
10,2026-11-11,01165 SUMMARECON EMERALD KARAWANG KWG,203.0,6.372492e+05,335880.4259,238558.2801,2.321630e+05,0,8.149748e+05,1.031285e+06,7.698172e+05,2.938272e+06,336904.5454,0,7.335105e+06
11,2026-11-12,01165 SUMMARECON EMERALD KARAWANG KWG,222.0,6.263063e+05,813153.1532,296407.8624,3.135366e+05,0,1.145504e+06,8.492591e+05,1.130333e+06,3.578784e+06,250368.1818,0,9.003652e+06
12,2026-11-13,01165 SUMMARECON EMERALD KARAWANG KWG,252.0,6.235790e+05,317272.7273,317028.3833,2.083175e+05,0,4.337203e+05,1.579647e+06,1.037925e+06,3.692832e+06,468263.6364,0,8.678586e+06
13,2026-11-14,01165 SUMMARECON EMERALD KARAWANG KWG,356.0,8.531245e+05,929519.2291,539717.1302,1.039112e+06,0,6.164843e+05,2.657179e+06,1.519170e+06,5.888579e+06,351240.9091,0,1.439413e+07
14,2026-11-15,01165 SUMMARECON EMERALD KARAWANG KWG,358.0,1.129914e+06,316363.6364,429273.5724,3.593175e+05,0,5.335844e+05,2.284334e+06,2.351694e+06,5.681064e+06,522672.7273,0,1.360822e+07


In [6]:
# Sanity check: bandingkan dengan Metabase (query 1) untuk bulan berjalan
this_month = datetime.now(timezone('Asia/Jakarta')).strftime('%Y-%m')
chk = detail[detail['date'].str.startswith(this_month)]
print('This month total_order :', f"{chk['total_order'].sum():,.0f}")
print('This month daily_income:', f"{chk['daily_income'].sum():,.0f}")

This month total_order : 3,665
This month daily_income: 139,218,977


### 3. Build Dashboard HTML

Data di-embed sebagai JSON ke setiap template di `DASHBOARDS` (tema ungu Jiwa Group dan tema Jiwa Light). Shape JSON ini juga yang nanti dikembalikan API kalau dashboard dijadikan web app live.

In [7]:
import base64

columns = ([{'key': 'date', 'label': 'Date', 'numeric': False},
            {'key': 'outlet_name', 'label': 'Outlet Name', 'numeric': False}]
           + [{'key': k, 'label': label, 'numeric': True} for k, label in PAYMENT_COLUMNS]
           + [{'key': 'daily_income', 'label': 'Daily Income', 'numeric': True}])

num_cols = ['total_order'] + BUCKETS + ['daily_income']
rows = detail.copy()
rows[num_cols] = rows[num_cols].round(4)

payload = {
    'outlet_code':  OUTLET_CODE,
    'outlet_title': OUTLET_TITLE,
    'lookback_days': LOOKBACK_DAYS,
    'data_start':   DATA_START,
    'generated_at': datetime.now(timezone('Asia/Jakarta')).strftime('%d %b %Y %H:%M'),
    'columns':      columns,
    'rows':         rows.to_dict('records'),
}

data_json = json.dumps(payload, ensure_ascii=False, separators=(',', ':')).replace('</', '<\\/')

# Logo Jiwa Group di-embed (base64) supaya tetap tampil di browser pihak landlord.
# jiwagroup.com memblokir hotlink gambar (Cross-Origin), jadi URL hanya dipakai sebagai cadangan.
JIWA_LOGO_URL = 'https://jiwagroup.com/assets/img/Jiwa-Group-Logo_JJ-PURPLE.png'
JIWA_LOGO_CANDIDATES = [Path('Jiwa-Group-Logo_JJ-PURPLE.png'), Path('assets') / 'Jiwa-Group-Logo_JJ-PURPLE.png']
JIWA_LOGO_FILE = next((f for f in JIWA_LOGO_CANDIDATES if f.exists()), None)
if JIWA_LOGO_FILE is not None:
    logo_src = 'data:image/png;base64,' + base64.b64encode(JIWA_LOGO_FILE.read_bytes()).decode()
    print('Logo :', 'embedded dari', JIWA_LOGO_FILE)
else:
    logo_src = JIWA_LOGO_URL
    print('Logo : pakai URL (simpan file logo di folder ini atau assets/ supaya di-embed)')

for template_path, output_path in DASHBOARDS:
    template = template_path.read_text(encoding='utf-8')
    assert '/*__DASHBOARD_DATA__*/null' in template, f'Placeholder data tidak ditemukan di {template_path}'
    html = (template
            .replace('__JIWA_LOGO_SRC__', logo_src)
            .replace('__PAGE_TITLE__', f'{OUTLET_TITLE} - Sales Dashboard')
            .replace('/*__DASHBOARD_DATA__*/null', data_json))
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(html, encoding='utf-8')
    print('Saved:', output_path.resolve(), f'({output_path.stat().st_size/1024:,.1f} KB)')

Logo : embedded dari Jiwa-Group-Logo_JJ-PURPLE.png
Saved: C:\GitHub\my code\api_integration_landlord\01165_sms_emerald_karawang\output\01165_Summarecon_Emerald_Karawang_Sales_Dashboard.html (111.8 KB)
Saved: C:\GitHub\my code\api_integration_landlord\01165_sms_emerald_karawang\output\01165_Summarecon_Emerald_Karawang_Sales_Dashboard_JiwaLight.html (66.5 KB)
